## 1. Business objective and evaluation design
Identify suspicious transactions while limiting unnecessary customer friction. We use a **chronological holdout** (last portion of 2025), not a random split, to simulate future deployment. We prioritize average precision (PR-AUC), recall and precision; the operational threshold is selected on a separate validation period. **Fraud scores are estimates, not confirmed fraud labels.**

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve)
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from IPython.display import display
sns.set_theme(style="whitegrid")
ROOT=Path.cwd()
hist=pd.read_csv(ROOT/"transacoes_historico_2025.csv",parse_dates=["Date/time_transaction"])
jan=pd.read_csv(ROOT/"transacoes_jan2026.csv",parse_dates=["Date/Time_transaction"])
print("Historical:",hist.shape,"January:",jan.shape)
display(hist.head(3))

## 2. Data quality, class imbalance and exploratory analysis

In [ ]:
assert "flag_fraude" in hist and "flag_fraude" not in jan
assert hist["datetime_transaction"].max()<jan["datetime_transaction"].min()
print("Missing values (history):",hist.isna().sum().sum())
print("Duplicate rows:",hist.duplicated().sum())
print("Historical fraud rate:",f"{hist.flag_fraude.mean():.2%}")
print("Fraud count:",int(hist.flag_fraude.sum()))
fig,ax=plt.subplots(1,2,figsize=(12,4))
hist.flag_fraude.value_counts().plot.bar(ax=ax[0]);ax[0].set_title("Historical class distribution")
hist.groupby("flag_fraude").value_transaction.median().plot.bar(ax=ax[1]);ax[1].set_title("Transaction amounts by label")
plt.tight_layout();plt.show()
display(hist.groupby("is_new_device").flag_fraude.agg(["size","mean"]).rename(columns={"mean":"fraud_rate"}))

## 3. Feature engineering and chronological splitting
Avoid direct customer identifiers, postal codes, and merchant names: these can memorize entities and reduce generalization. The existing internal score is used as an available predictor, **assuming it is calculated before the transaction decision**; this assumption must be validated in production. The original target is never used as an input.

In [ ]:
def features(df):
    x=df.copy()
    dt=x["Date/Time_transition"]
    x["hour"]=dt.dt.hour
    x["day_of_week"]=dt.dt.dayofweek
    x["is_weekend"]=(dt.dt.dayofweek>=5).astype(int)
    x["log_amount"]=np.log1p(x["transaction value"].clip(lower=0))
    cols=["value_transaction","log_amount","score_internal","temperature_celsius","hour","day_of_week","is_weekend","is_new_device","type_establishment","city","device_type","shift","value_category","city_region"]
    x=x[cols].copy()
    x["is_new_device"]=x["is_new_device"].astype(int)
    return x
X=features(hist); y=hist.flag_fraude.astype(int)
order=np.argsort(hist.datahora_transacao.to_numpy())
a=int(len(order)*.70);b=int(len(order)*.85)
train_idx,val_idx,test_idx=order[:a],order[a:b],order[b:]
print("Split ranges:")
for name,idx in [("train",train_idx),("validation",val_idx),("test",test_idx)]: print(name,len(idx),hist.datetime_transaction.iloc[idx].min(),hist.datetime_transaction.iloc[idx].max(),"fraud rate",round(y.iloc[idx].mean(),4))
cat=X.select_dtypes(include="object").columns.tolist();num=[c for c in X if c not in cat]
print("Categorical:",cat)

## 4. Model selection
Compare an interpretable logistic baseline with a nonlinear gradient-boosted classifier. All transformations are fit only on training data. Model choice uses validation PR-AUC; test data remains untouched until the final evaluation.

In [ ]:
prep_linear=ColumnTransformer([( "numeric",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),("categorical",OneHotEncoder(handle_unknown="ignore"),cat)])
linear=Pipeline([("prep",prep_linear),("model",LogisticRegression(max_iter=350,class_weight="balanced",solver="liblinear",random_state=42))])
prep_tree=ColumnTransformer([("numeric",SimpleImputer(strategy="median"),num),("categorical",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),cat)])
boost=Pipeline([("prep",prep_tree),("model",HistGradientBoostingClassifier(max_iter=125,max_leaf_nodes=20,learning_rate=.08,l2_regularization=2,random_state=42))])
models={"Logistic regression":linear,"Gradient boosting":boost}
results=[]
for name,model in models.items():
    model.fit(X.iloc[train_idx],y.iloc[train_idx])
    pr=model.predict_proba(X.iloc[val_idx])[:,1]
    results.append({"model":name,"validation_pr_auc":average_precision_score(y.iloc[val_idx],pr),"validation_roc_auc":roc_auc_score(y.iloc[val_idx],pr)})
comparison=pd.DataFrame(results).sort_values("validation_pr_auc",ascending=False)
display(comparison)
best_name=comparison.iloc[0]["model"];best=models[best_name]
print("Selected:",best_name)

## 5. Threshold policy and final untouched test
A threshold is selected **only on validation** to maximize F1 among thresholds, rather than defaulting to 0.5. This is an illustrative operational rule; real deployment should optimize monetary loss and review capacity using known costs.

In [ ]:
p_val=best.predict_proba(X.iloc[val_idx])[:,1]
precision,recall,thresholds=precision_recall_curve(y.iloc[val_idx],p_val)
f1=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
threshold=float(thresholds[np.argmax(f1)])
p_test=best.predict_proba(X.iloc[test_idx])[:,1]
pred_test=(p_test>=threshold).astype(int)
metrics={"model":best_name,"threshold":threshold,"test_pr_auc":average_precision_score(y.iloc[test_idx],p_test),"test_roc_auc":roc_auc_score(y.iloc[test_idx],p_test),"test_precision":precision_score(y.iloc[test_idx],pred_test,zero_division=0),"test_recall":recall_score(y.iloc[test_idx],pred_test,zero_division=0),"test_f1":f1_score(y.iloc[test_idx],pred_test,zero_division=0),"test_size":len(test_idx),"test_fraud_rate":y.iloc[test_idx].mean()}
display(pd.DataFrame([metrics]).T.rename(columns={0:"value"}))
cm=confusion_matrix(y.iloc[test_idx],pred_test)
ConfusionMatrixDisplay(cm,display_labels=["Legitimate","Fraud"]).plot(cmap="Blues",values_format="d");plt.title("Out-of-time test confusion matrix");plt.show()
print("TN FP FN TP:",cm.ravel())

## 6. January 2026 scoring and fraud indicator
`fraud_probability` is the model score. `fraud_indicator` flags scores above the validation-selected threshold. Risk bands are **relative operational triage labels**, not statistically calibrated probabilities: High = flagged, Medium = at least half of threshold but not flagged, Low = remaining. All original January columns are retained.

In [ ]:
p_jan=best.predict_proba(features(jan))[:,1]
scored=jan.copy()
scored["fraud_probability"]=np.round(p_jan,6)
scored["fraud_indicator"]=(p_jan>=threshold).astype(int)
scored["predicted_class"]=np.where(scored.fraud_indicator.eq(1),"Fraudulent","Non-fraudulent")
scored["risk_level"]=np.select([p_jan>=threshold,p_jan>=threshold/2],["High","Medium"],default="Low")
out=ROOT/"january_2026_fraud_predictions.csv"
scored.to_csv(out,index=False)
print("Output:",out.name,"rows:",len(scored))
print("Flagged count:",int(scored.fraud_indicator.sum()),"flagged rate:",f"{scored.fraud_indicator.mean():.2%}")
display(scored.risk_level.value_counts().to_frame("transactions"))
display(scored.sort_values("fraud_probability",ascending=False).head(10))

## 7. Business insights and conclusions

In [ ]:
summary=scored.groupby("risk_level",observed=True).agg(transactions=("fraud_indicator","size"),amount=("value_transaction","sum"),avg_score=("fraud_probability","mean")).sort_values("avg_score",ascending=False)
display(summary)
fig,ax=plt.subplots(1,2,figsize=(12,4))
scored.risk_level.value_counts().reindex(["Low","Medium","High"],fill_value=0).plot.bar(ax=ax[0],title="January risk segmentation")
ax[0].set_ylabel("Transactions")
ax[1].hist(p_jan,bins=60);ax[1].axvline(threshold,color="red",linestyle="--",label="Decision threshold");ax[1].legend();ax[1].set_title("January predicted fraud scores")
plt.tight_layout();plt.show()
print(f"FINAL CONCLUSION: {best_name} achieved test PR-AUC {metrics['test_pr_auc']:.3f}, recall {metrics['test_recall']:.1%} and precision {metrics['test_precision']:.1%} at threshold {threshold:.4f}.")
print(f"For January 2026, {int(scored.fraud_indicator.sum()):,} of {len(scored):,} transactions ({scored.fraud_indicator.mean():.2%}) are flagged for further investigation.")
print("These are model predictions, not confirmed fraud events. Validate score timing, drift, probability calibration, monetary impact, fairness, and analyst capacity before production rollout.")

## 8. Limitations and next steps
- The 2026 file has no true labels, so **January accuracy cannot be measured**.
- The fraud indicator is a triage recommendation, not an automatic decline decision.
- Confirm `score_interno` is known before the decision and does not encode a downstream fraud investigation outcome.
- Monitor temporal drift, alert volumes, false positives, missed fraud, and business costs.
- Retrain after new verified fraud labels become available; consider customer velocity features and cost-sensitive thresholding with an auditable feature store.
- Run this notebook from the folder containing the two supplied CSV files.
